# Notebook 24 – Explainability Intelligence Engine

## Financial Fraud Detection System

---

## Objective

This notebook provides **Explainable AI (XAI)** capabilities for the deployed **Random Forest (Class Weights)** model.

Unlike traditional feature importance, this notebook generates **transaction-level explainability** that can be directly consumed by the Power BI **AI Decision Center**.

---

## Inputs

- `models/best_model.pkl`
- `data/processed/X_test.csv`
- `data/processed/y_test.csv`

---

## Outputs

`powerbi_assets/explainability/`

- transaction_explainability.csv
- feature_importance_shap.csv
- executive_summary.json
- model_card.json
- shap_beeswarm.png
- global_feature_importance.png

---

### Author

Ahmad Yaseen

### Project

Financial Fraud Detection System

## Import Libraries

In [1]:
# =====================================================
# Import Libraries
# =====================================================

import os
import json
import warnings
from datetime import datetime

import joblib
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

import shap

warnings.filterwarnings("ignore")

/Users/yaseen/Data_Analysis/Project_Files/Financial_Fraud_Detection/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Project Banner

In [2]:
print("=" * 70)
print("FINANCIAL FRAUD DETECTION SYSTEM")
print("Explainability Intelligence Engine")
print("=" * 70)
print()
print("Loading Project...")

FINANCIAL FRAUD DETECTION SYSTEM
Explainability Intelligence Engine

Loading Project...


## Configuration

In [3]:
# =====================================================
# Project Configuration
# =====================================================

MODEL_PATH = "../models/best_model.pkl"

X_TEST_PATH = "../data/processed/X_test.csv"

Y_TEST_PATH = "../data/processed/y_test.csv"

OUTPUT_DIR = "../powerbi_assets/explainability"

RANDOM_STATE = 42

MAX_SHAP_SAMPLES = 1000

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Configuration Loaded")

Configuration Loaded


## Verify Project Files

In [4]:
# =====================================================
# Verify Required Files
# =====================================================

required_files = {

    "Model": MODEL_PATH,

    "X Test": X_TEST_PATH,

    "y Test": Y_TEST_PATH

}

print("=" * 70)
print("Checking Project Files")
print("=" * 70)

missing = []

for name, path in required_files.items():

    if os.path.exists(path):

        print(f"✓ {name}")

    else:

        print(f"✗ {name}")

        missing.append(path)

if missing:

    raise FileNotFoundError(

        f"\nMissing Files:\n" +

        "\n".join(missing)

    )

print()

print("All Required Files Found")

Checking Project Files
✓ Model
✓ X Test
✓ y Test

All Required Files Found


## Load Production Model

In [5]:
print("=" * 70)
print("Loading Production Model")
print("=" * 70)

model = joblib.load(MODEL_PATH)

print(type(model))

print()

print("Production Model Loaded Successfully")

Loading Production Model
<class 'sklearn.ensemble._forest.RandomForestClassifier'>

Production Model Loaded Successfully


## Load Test Dataset

In [6]:
print("=" * 70)
print("Loading Test Dataset")
print("=" * 70)

X_test = pd.read_csv(X_TEST_PATH)

y_test = pd.read_csv(Y_TEST_PATH)

y_test = y_test.squeeze()

print()

print("X_test Shape :", X_test.shape)

print("y_test Shape :", y_test.shape)

Loading Test Dataset

X_test Shape : (56746, 33)
y_test Shape : (56746,)


## Dataset Validation

In [7]:
print("=" * 70)
print("Dataset Validation")
print("=" * 70)

print()

print("Missing Values")

print(X_test.isnull().sum().sum())

print()

print("Duplicate Rows")

print(X_test.duplicated().sum())

print()

print("Feature Count")

print(len(X_test.columns))

print()

print("Target Distribution")

print(y_test.value_counts())

Dataset Validation

Missing Values
0

Duplicate Rows
0

Feature Count
33

Target Distribution
Class
0    56651
1       95
Name: count, dtype: int64


## Dataset Preview

In [8]:
display(X_test.head())

display(y_test.head())

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V23,V24,V25,V26,V27,V28,Amount,Large_Transaction,Log_Amount,Amount_Level
0,-0.705990,1.228821,-0.063408,0.274145,0.647465,-0.048135,0.372073,-0.224231,0.079939,0.640759,...,-0.151661,-0.700372,0.598550,0.491409,0.002989,0.001782,-0.307400,0,4.683427,3
1,1.275941,-0.203154,1.176678,-0.759595,-0.518472,0.629649,-0.721675,0.638893,0.243377,-0.157488,...,-0.082753,0.508386,-0.710906,-0.234510,0.379640,0.261351,-0.345579,0,2.388763,1
2,-1.346436,-1.672836,1.401297,1.503940,2.175491,0.699791,1.062139,1.114364,-0.535822,-0.252983,...,-0.280083,-0.846468,-0.155456,-0.062383,0.007777,0.113900,0.011211,0,4.799091,3
3,-0.876311,0.819379,-1.124913,0.515025,0.513945,-1.009048,0.488484,-0.580672,0.187686,-0.999142,...,-0.198732,-0.337408,0.238538,-0.289454,0.038214,0.058407,0.557220,0,2.150599,1
4,0.729091,2.009701,0.105635,-1.752759,0.588312,0.374801,-0.637884,0.009260,-0.129487,0.492619,...,-0.013472,-0.446920,0.111522,0.642944,-0.036998,-0.043404,-0.347696,0,0.688135,0


0    0
1    0
2    0
3    0
4    0
Name: Class, dtype: int64

## Model Prediction

In [9]:
print("=" * 70)
print("Generating Predictions")
print("=" * 70)

predictions = model.predict(X_test)

probabilities = model.predict_proba(X_test)[:, 1]

print()

print("Predictions Generated Successfully")

Generating Predictions

Predictions Generated Successfully


## Prediction Summary

In [10]:
prediction_results = pd.DataFrame({

    "Actual": y_test,

    "Prediction": predictions,

    "Fraud_Probability": probabilities

})

display(prediction_results.head())

print()

print(prediction_results.describe())

,Actual,Prediction,Fraud_Probability
0,0,0,0.0
1,0,0,0.0
2,0,0,0.0
3,0,0,0.0
4,0,0,0.0



             Actual    Prediction  Fraud_Probability
count  56746.000000  56746.000000       56746.000000
mean       0.001674      0.001269           0.001881
std        0.040882      0.035598           0.033323
min        0.000000      0.000000           0.000000
25%        0.000000      0.000000           0.000000
50%        0.000000      0.000000           0.000000
75%        0.000000      0.000000           0.000000
max        1.000000      1.000000           1.000000


## Save Prediction Sample

In [11]:
prediction_results.head(100).to_csv(

    os.path.join(

        OUTPUT_DIR,

        "prediction_sample.csv"

    ),

    index=False

)

print("Prediction Sample Saved")

Prediction Sample Saved


## Notebook Status

In [12]:
print("=" * 70)
print("PART 1 COMPLETED")
print("=" * 70)

print()

print("✓ Project Configuration")

print("✓ Model Loaded")

print("✓ Dataset Loaded")

print("✓ Dataset Validated")

print("✓ Predictions Generated")

print("✓ Prediction Sample Exported")

PART 1 COMPLETED

✓ Project Configuration
✓ Model Loaded
✓ Dataset Loaded
✓ Dataset Validated
✓ Predictions Generated
✓ Prediction Sample Exported


## Select Sample for SHAP

In [13]:
# =====================================================
# Select Sample for SHAP Analysis
# =====================================================

print("=" * 70)
print("Preparing SHAP Dataset")
print("=" * 70)

sample_size = min(MAX_SHAP_SAMPLES, len(X_test))

X_shap = (

    X_test

    .sample(

        n=sample_size,

        random_state=RANDOM_STATE

    )

    .reset_index(drop=True)

)

print(f"Sample Size : {len(X_shap):,}")

print(f"Feature Count : {X_shap.shape[1]}")

Preparing SHAP Dataset
Sample Size : 1,000
Feature Count : 33


## Initialize SHAP Explainer

In [14]:
# =====================================================
# Initialize SHAP Tree Explainer
# =====================================================

print("=" * 70)
print("Initializing SHAP TreeExplainer")
print("=" * 70)

explainer = shap.TreeExplainer(model)

print("TreeExplainer Ready")

Initializing SHAP TreeExplainer
TreeExplainer Ready


## Compute SHAP Values

In [15]:
# =====================================================
# Compute SHAP Values
# =====================================================

print("=" * 70)
print("Computing SHAP Values")
print("=" * 70)

raw_shap = explainer.shap_values(X_shap)

print("Raw SHAP Values Generated")

Computing SHAP Values
Raw SHAP Values Generated


## Normalize SHAP Output

In [16]:
# =====================================================
# Normalize SHAP Output
# =====================================================

if isinstance(raw_shap, list):

    # Older SHAP versions
    shap_values = raw_shap[1]

elif isinstance(raw_shap, np.ndarray):

    if raw_shap.ndim == 3:

        # Shape = (samples, features, classes)
        # Use Fraud class (class 1)
        shap_values = raw_shap[:, :, 1]

    else:

        shap_values = raw_shap

elif hasattr(raw_shap, "values"):

    shap_values = raw_shap.values

    if shap_values.ndim == 3:

        shap_values = shap_values[:, :, 1]

else:

    raise ValueError("Unsupported SHAP output format")

print("Final SHAP Shape:")

print(shap_values.shape)

Final SHAP Shape:
(1000, 33)


## Global Feature Importance

In [17]:
# =====================================================
# Global SHAP Feature Importance
# =====================================================

importance = np.abs(shap_values).mean(axis=0)

feature_importance = (

    pd.DataFrame({

        "Feature": X_shap.columns,

        "Mean_SHAP": importance

    })

    .sort_values(

        "Mean_SHAP",

        ascending=False

    )

    .reset_index(drop=True)

)

display(feature_importance.head(15))

,Feature,Mean_SHAP
0,V12,0.074165
1,V14,0.071306
2,V3,0.053889
3,V4,0.050054
4,V10,0.048554
5,V11,0.044596
6,V17,0.034397
7,V16,0.016949
8,V7,0.014647
9,V1,0.009177


## Save Feature Importance

In [18]:
feature_importance.to_csv(

    os.path.join(

        OUTPUT_DIR,

        "feature_importance_shap.csv"

    ),

    index=False

)

print("feature_importance_shap.csv Saved")

feature_importance_shap.csv Saved


## SHAP Matrix

In [19]:
shap_matrix = pd.DataFrame(

    shap_values,

    columns=X_shap.columns

)

display(shap_matrix.head())

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V23,V24,V25,V26,V27,V28,Amount,Large_Transaction,Log_Amount,Amount_Level
0,-0.003740,-0.003292,-0.009776,-0.015383,-0.044041,-0.004610,-0.012668,-0.015877,-0.010356,-0.005489,...,-0.001586,-0.014926,-0.003292,-0.004862,-0.003575,-0.001278,-0.005487,-0.000044,-0.002395,-0.000701
1,-0.002459,-0.005949,-0.006350,-0.064096,-0.022913,-0.005750,-0.001694,-0.012291,-0.002278,-0.005020,...,-0.001976,-0.003606,-0.002815,-0.004410,-0.003319,-0.001470,-0.004075,-0.000782,-0.008482,-0.001155
2,-0.006510,-0.004284,-0.007342,-0.012688,-0.088236,-0.008332,-0.003224,-0.026527,0.008597,-0.002825,...,-0.002821,-0.003352,-0.004180,-0.017253,-0.006030,-0.002619,0.000938,-0.000031,-0.004046,-0.001268
3,-0.002799,-0.003000,-0.003455,-0.078686,-0.034982,-0.003745,-0.001627,-0.013223,-0.002939,-0.004839,...,-0.002025,-0.004192,-0.002044,-0.004138,-0.006509,-0.001943,-0.004201,-0.000037,-0.002489,-0.000751
4,-0.021370,-0.004839,-0.006531,-0.144435,0.050921,-0.003627,-0.000688,-0.016672,-0.016082,-0.004172,...,-0.001842,-0.002010,-0.008754,-0.005642,-0.013376,-0.008386,-0.006552,-0.000025,-0.007649,-0.001610


## Export SHAP Matrix

In [20]:
shap_matrix.to_csv(

    os.path.join(

        OUTPUT_DIR,

        "shap_matrix.csv"

    ),

    index=False

)

print("shap_matrix.csv Saved")

shap_matrix.csv Saved


## Global SHAP Bar Plot

In [21]:
plt.figure(figsize=(10,8))

shap.summary_plot(

    shap_values,

    X_shap,

    plot_type="bar",

    show=False

)

plt.tight_layout()

plt.savefig(

    os.path.join(

        OUTPUT_DIR,

        "global_feature_importance.png"

    ),

    dpi=300,

    bbox_inches="tight"

)

plt.close()

print("Global Feature Importance Saved")

Global Feature Importance Saved


## SHAP Beeswarm

In [22]:
plt.figure(figsize=(12,8))

shap.summary_plot(

    shap_values,

    X_shap,

    show=False

)

plt.tight_layout()

plt.savefig(

    os.path.join(

        OUTPUT_DIR,

        "shap_beeswarm.png"

    ),

    dpi=300,

    bbox_inches="tight"

)

plt.close()

print("SHAP Beeswarm Saved")

SHAP Beeswarm Saved


## Executive Statistics

In [23]:
summary_stats = {

    "Transactions_Explained": int(len(X_shap)),

    "Total_Features": int(X_shap.shape[1]),

    "Average_SHAP": float(np.abs(shap_values).mean()),

    "Maximum_SHAP": float(np.abs(shap_values).max()),

    "Generated_On": datetime.now().strftime("%Y-%m-%d %H:%M:%S")

}

summary_stats

{'Transactions_Explained': 1000,
 'Total_Features': 33,
 'Average_SHAP': 0.015690234313009123,
 'Maximum_SHAP': 0.181250046389347,
 'Generated_On': '2026-07-20 12:01:24'}

## Save Executive Statistics

In [24]:
with open(

    os.path.join(

        OUTPUT_DIR,

        "executive_summary.json"

    ),

    "w"

) as f:

    json.dump(summary_stats, f, indent=4)

print("Executive Summary Saved")

Executive Summary Saved


## Notebook Status

In [25]:
print("=" * 70)
print("PART 2 COMPLETED")
print("=" * 70)

generated = [

    "feature_importance_shap.csv",

    "shap_matrix.csv",

    "global_feature_importance.png",

    "shap_beeswarm.png",

    "executive_summary.json"

]

for item in generated:

    print(f"✓ {item}")

PART 2 COMPLETED
✓ feature_importance_shap.csv
✓ shap_matrix.csv
✓ global_feature_importance.png
✓ shap_beeswarm.png
✓ executive_summary.json


In [26]:
feature_importance = (
    pd.DataFrame({
        "Feature": X_shap.columns,
        "Mean_SHAP": importance
    })
    .sort_values("Mean_SHAP", ascending=False)
    .reset_index(drop=True)
)

display(feature_importance.head(15))

,Feature,Mean_SHAP
0,V12,0.074165
1,V14,0.071306
2,V3,0.053889
3,V4,0.050054
4,V10,0.048554
5,V11,0.044596
6,V17,0.034397
7,V16,0.016949
8,V7,0.014647
9,V1,0.009177


## Generate SHAP Values for All Transactions

In [27]:
# =====================================================
# Generate SHAP Values for Full Test Dataset
# =====================================================

print("=" * 70)
print("Generating SHAP Values for Full Test Dataset")
print("=" * 70)

raw_shap_full = explainer.shap_values(X_test)

if isinstance(raw_shap_full, list):

    shap_full = raw_shap_full[1]

elif isinstance(raw_shap_full, np.ndarray):

    if raw_shap_full.ndim == 3:
        shap_full = raw_shap_full[:, :, 1]
    else:
        shap_full = raw_shap_full

elif hasattr(raw_shap_full, "values"):

    shap_full = raw_shap_full.values

    if shap_full.ndim == 3:
        shap_full = shap_full[:, :, 1]

else:
    raise ValueError("Unsupported SHAP Output")

print("Full SHAP Shape :", shap_full.shape)

Generating SHAP Values for Full Test Dataset
Full SHAP Shape : (56746, 33)


## Risk Level Function

In [28]:
# =====================================================
# Risk Classification
# =====================================================

def classify_risk(probability):

    if probability >= 0.90:
        return "Critical"

    elif probability >= 0.70:
        return "High"

    elif probability >= 0.40:
        return "Medium"

    return "Low"

## Recommendation Function

In [29]:
# =====================================================
# Business Recommendation
# =====================================================

def recommendation(probability):

    if probability >= 0.90:
        return "Block Transaction"

    elif probability >= 0.70:
        return "Manual Investigation"

    elif probability >= 0.40:
        return "Monitor Customer"

    return "Approve Transaction"

## Top Feature Extraction Function

In [30]:
# =====================================================
# Extract Top SHAP Features
# =====================================================

def get_top_features(values, feature_names, top_n=5):

    values = np.array(values)

    idx = np.argsort(np.abs(values))[::-1][:top_n]

    return [

        (

            feature_names[i],

            values[i]

        )

        for i in idx

    ]

## Build Explainability Records

In [31]:
# =====================================================
# Generate Transaction Explainability
# =====================================================

print("=" * 70)
print("Generating Explainability Records")
print("=" * 70)

records = []

feature_names = list(X_test.columns)

for i in range(len(X_test)):

    top_features = get_top_features(

        shap_full[i],

        feature_names

    )

    record = {

        "Transaction_ID": f"TRX-{i+1:06d}",

        "Actual": int(y_test.iloc[i]),

        "Prediction": int(predictions[i]),

        "Fraud_Probability": round(float(probabilities[i]), 6),

        "Risk_Level": classify_risk(probabilities[i]),

        "Recommendation": recommendation(probabilities[i])

    }

    for j, (feature, value) in enumerate(top_features, start=1):

        record[f"Top_Feature_{j}"] = feature

        record[f"Contribution_{j}"] = round(float(value), 6)

    records.append(record)

print(f"Generated {len(records):,} Records")

Generating Explainability Records
Generated 56,746 Records


## Create Explainability DataFrame

In [32]:
transaction_explainability = pd.DataFrame(records)

display(transaction_explainability.head())

,Transaction_ID,Actual,Prediction,Fraud_Probability,Risk_Level,Recommendation,Top_Feature_1,Contribution_1,Top_Feature_2,Contribution_2,Top_Feature_3,Contribution_3,Top_Feature_4,Contribution_4,Top_Feature_5,Contribution_5
0,TRX-000001,0,0,0.0,Low,Approve Transaction,V12,-0.082708,V11,-0.076246,V3,-0.065172,V14,-0.062817,V10,-0.040793
1,TRX-000002,0,0,0.0,Low,Approve Transaction,V11,-0.092674,V14,-0.075934,V12,-0.071266,V10,-0.046229,V4,-0.045034
2,TRX-000003,0,0,0.0,Low,Approve Transaction,V3,-0.102495,V10,-0.091733,V12,-0.080081,V14,-0.048524,V4,0.037371
3,TRX-000004,0,0,0.0,Low,Approve Transaction,V12,-0.091367,V14,-0.076734,V3,-0.075118,V10,-0.056169,V17,-0.033981
4,TRX-000005,0,0,0.0,Low,Approve Transaction,V12,-0.121237,V10,-0.053267,V1,-0.037587,V4,-0.035067,V17,-0.031866


## Distribution Summary

In [33]:
print("=" * 70)
print("Risk Distribution")
print("=" * 70)

display(

    transaction_explainability["Risk_Level"]

    .value_counts()

)

Risk Distribution


Risk_Level
Low         56669
Critical       47
High           21
Medium          9
Name: count, dtype: int64

## Recommendation Summary

In [34]:
print("=" * 70)
print("Recommendation Distribution")
print("=" * 70)

display(

    transaction_explainability["Recommendation"]

    .value_counts()

)

Recommendation Distribution


Recommendation
Approve Transaction     56669
Block Transaction          47
Manual Investigation       21
Monitor Customer            9
Name: count, dtype: int64

## Export Explainability Dataset

In [35]:
transaction_explainability.to_csv(

    os.path.join(

        OUTPUT_DIR,

        "transaction_explainability.csv"

    ),

    index=False

)

print("transaction_explainability.csv Saved")

transaction_explainability.csv Saved


## Preview High-Risk Transactions

In [36]:
high_risk = transaction_explainability[

    transaction_explainability["Risk_Level"] == "Critical"

]

display(high_risk.head(10))

,Transaction_ID,Actual,Prediction,Fraud_Probability,Risk_Level,Recommendation,Top_Feature_1,Contribution_1,Top_Feature_2,Contribution_2,Top_Feature_3,Contribution_3,Top_Feature_4,Contribution_4,Top_Feature_5,Contribution_5
845,TRX-000846,1,1,0.90,Critical,Block Transaction,V10,0.116148,V14,0.109975,V17,0.109755,V12,0.081931,V4,0.033820
5638,TRX-005639,1,1,1.00,Critical,Block Transaction,V14,0.116129,V17,0.102525,V12,0.091128,V10,0.089801,V4,0.040510
5759,TRX-005760,1,1,0.98,Critical,Block Transaction,V14,0.125307,V10,0.104392,V17,0.089750,V12,0.088678,V4,0.038309
6673,TRX-006674,1,1,1.00,Critical,Block Transaction,V14,0.114988,V10,0.090540,V17,0.089788,V12,0.084799,V3,0.039122
8815,TRX-008816,1,1,0.98,Critical,Block Transaction,V14,0.151010,V10,0.150484,V12,0.097770,V4,0.071649,V11,0.047381
9744,TRX-009745,1,1,0.99,Critical,Block Transaction,V14,0.114118,V17,0.096574,V10,0.095396,V12,0.090896,V4,0.035004
11046,TRX-011047,1,1,0.97,Critical,Block Transaction,V14,0.140252,V10,0.118304,V17,0.116375,V12,0.092676,V4,0.042616
12091,TRX-012092,1,1,0.97,Critical,Block Transaction,V14,0.125258,V10,0.110984,V12,0.085627,V17,0.080090,V4,0.041366
14414,TRX-014415,1,1,1.00,Critical,Block Transaction,V14,0.126336,V10,0.102846,V12,0.097863,V17,0.071609,V4,0.046833
15092,TRX-015093,1,1,0.98,Critical,Block Transaction,V14,0.112142,V17,0.102004,V10,0.095115,V12,0.089597,V4,0.035563


## Part 3 Completed

In [37]:
print("=" * 70)
print("PART 3 COMPLETED")
print("=" * 70)

print()

print(f"Transactions Explained : {len(transaction_explainability):,}")

print()

print("Output File")

print("transaction_explainability.csv")

print()

print("Ready for Power BI Integration")

PART 3 COMPLETED

Transactions Explained : 56,746

Output File
transaction_explainability.csv

Ready for Power BI Integration


## Risk Distribution

In [38]:
# =====================================================
# Risk Distribution Summary
# =====================================================

risk_summary = (

    transaction_explainability["Risk_Level"]

    .value_counts()

    .rename_axis("Risk_Level")

    .reset_index(name="Count")

)

display(risk_summary)

,Risk_Level,Count
0,Low,56669
1,Critical,47
2,High,21
3,Medium,9


## Recommendation Distribution

In [39]:
# =====================================================
# Recommendation Summary
# =====================================================

recommendation_summary = (

    transaction_explainability["Recommendation"]

    .value_counts()

    .rename_axis("Recommendation")

    .reset_index(name="Count")

)

display(recommendation_summary)

,Recommendation,Count
0,Approve Transaction,56669
1,Block Transaction,47
2,Manual Investigation,21
3,Monitor Customer,9


## Save Summary CSVs

In [40]:
risk_summary.to_csv(

    os.path.join(

        OUTPUT_DIR,

        "risk_distribution.csv"

    ),

    index=False

)

recommendation_summary.to_csv(

    os.path.join(

        OUTPUT_DIR,

        "recommendation_distribution.csv"

    ),

    index=False

)

print("Summary CSV Files Saved")

Summary CSV Files Saved


## Build Executive Summary

In [41]:
# =====================================================
# Executive Dashboard Summary
# =====================================================

executive_summary = {

    "Model": "Random Forest (Class Weights)",

    "Algorithm": "Random Forest",

    "Transactions": int(len(transaction_explainability)),

    "Fraud_Predictions": int((prediction_results["Prediction"] == 1).sum()),

    "Genuine_Predictions": int((prediction_results["Prediction"] == 0).sum()),

    "Critical_Risk": int((transaction_explainability["Risk_Level"] == "Critical").sum()),

    "High_Risk": int((transaction_explainability["Risk_Level"] == "High").sum()),

    "Medium_Risk": int((transaction_explainability["Risk_Level"] == "Medium").sum()),

    "Low_Risk": int((transaction_explainability["Risk_Level"] == "Low").sum()),

    "Average_Fraud_Probability":

        round(

            float(

                prediction_results["Fraud_Probability"].mean()

            ),

            6

        ),

    "Generated_On":

        datetime.now().strftime("%Y-%m-%d %H:%M:%S")

}

executive_summary

{'Model': 'Random Forest (Class Weights)',
 'Algorithm': 'Random Forest',
 'Transactions': 56746,
 'Fraud_Predictions': 72,
 'Genuine_Predictions': 56674,
 'Critical_Risk': 47,
 'High_Risk': 21,
 'Medium_Risk': 9,
 'Low_Risk': 56669,
 'Average_Fraud_Probability': 0.001881,
 'Generated_On': '2026-07-20 12:05:02'}

## Export Executive Summary

In [42]:
with open(

    os.path.join(

        OUTPUT_DIR,

        "executive_dashboard_summary.json"

    ),

    "w"

) as f:

    json.dump(

        executive_summary,

        f,

        indent=4

    )

print("Executive Dashboard Summary Saved")

Executive Dashboard Summary Saved


## Build Model Card

In [43]:
# =====================================================
# Model Card
# =====================================================

model_card = {

    "Project":

        "Financial Fraud Detection System",

    "Model":

        "Random Forest (Class Weights)",

    "Algorithm":

        "Random Forest",

    "Problem":

        "Binary Classification",

    "Target":

        "Fraud Detection",

    "Features":

        int(X_test.shape[1]),

    "Transactions_Explained":

        int(len(transaction_explainability)),

    "SHAP_Version":

        shap.__version__,

    "Generated_On":

        datetime.now().strftime("%Y-%m-%d %H:%M:%S"),

    "Author":

        "Ahmad Yaseen"

}

model_card

{'Project': 'Financial Fraud Detection System',
 'Model': 'Random Forest (Class Weights)',
 'Algorithm': 'Random Forest',
 'Problem': 'Binary Classification',
 'Target': 'Fraud Detection',
 'Features': 33,
 'Transactions_Explained': 56746,
 'SHAP_Version': '0.52.0',
 'Generated_On': '2026-07-20 12:05:02',
 'Author': 'Ahmad Yaseen'}

## Export Model Card

In [44]:
with open(

    os.path.join(

        OUTPUT_DIR,

        "model_card.json"

    ),

    "w"

) as f:

    json.dump(

        model_card,

        f,

        indent=4

    )

print("Model Card Saved")

Model Card Saved


## Export Top 10 Global Features

In [45]:
top10_features = feature_importance.head(10)

top10_features.to_csv(

    os.path.join(

        OUTPUT_DIR,

        "top10_global_features.csv"

    ),

    index=False

)

display(top10_features)

,Feature,Mean_SHAP
0,V12,0.074165
1,V14,0.071306
2,V3,0.053889
3,V4,0.050054
4,V10,0.048554
5,V11,0.044596
6,V17,0.034397
7,V16,0.016949
8,V7,0.014647
9,V1,0.009177


## Output Folder Verification

In [46]:
# =====================================================
# Generated Assets
# =====================================================

generated_files = sorted(os.listdir(OUTPUT_DIR))

print("=" * 70)
print("Generated Explainability Assets")
print("=" * 70)

for file in generated_files:

    print(f"✓ {file}")

Generated Explainability Assets
✓ business_recommendations.csv
✓ executive_dashboard_summary.json
✓ executive_summary.json
✓ feature_contributions_long.csv
✓ feature_importance_shap.csv
✓ global_feature_importance.png
✓ local_explanations.csv
✓ model_card.json
✓ prediction_sample.csv
✓ recommendation_distribution.csv
✓ risk_distribution.csv
✓ shap_bar.png
✓ shap_beeswarm.png
✓ shap_matrix.csv
✓ shap_summary.csv
✓ top10_global_features.csv
✓ top20_features.csv
✓ transaction_explainability.csv
✓ waterfall_example.png


## Final Notebook Summary

In [47]:
print("=" * 70)
print("NOTEBOOK 24 COMPLETED")
print("=" * 70)

print()

print("Explainability Intelligence Engine Successfully Built")

print()

print(f"Transactions Explained : {len(transaction_explainability):,}")

print(f"Features : {X_test.shape[1]}")

print(f"Global Features Ranked : {len(feature_importance)}")

print()

print("Power BI Assets Ready")

print()

print("✓ Global Feature Importance")

print("✓ SHAP Beeswarm")

print("✓ Feature Importance CSV")

print("✓ Transaction Explainability")

print("✓ Risk Distribution")

print("✓ Recommendation Distribution")

print("✓ Executive Dashboard Summary")

print("✓ Model Card")

print("✓ Top 10 Global Features")

print()

print("Ready for Power BI AI Decision Center")

NOTEBOOK 24 COMPLETED

Explainability Intelligence Engine Successfully Built

Transactions Explained : 56,746
Features : 33
Global Features Ranked : 33

Power BI Assets Ready

✓ Global Feature Importance
✓ SHAP Beeswarm
✓ Feature Importance CSV
✓ Transaction Explainability
✓ Risk Distribution
✓ Recommendation Distribution
✓ Executive Dashboard Summary
✓ Model Card
✓ Top 10 Global Features

Ready for Power BI AI Decision Center
